[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mateus-Gabardo/bend2-verification-impl/blob/main/playground.ipynb)

# Playground: reserva de estoque em Bend 2

Este notebook é uma versão simplificada do estudo do artigo *Características Funcionais na Verificação de Propriedades em Bend 2*. Aqui o código Bend fica **dentro das células**: você pode ler, alterar e executar na hora.

**Como usar:**

1. Execute a célula 1 (▶ ou Shift+Enter). Ela instala o Bend e prepara o notebook.
2. Execute as demais células na ordem.
3. Para experimentar, **edite o código de qualquer célula e aperte Shift+Enter de novo**. O resultado aparece logo abaixo.

Cada célula de código começa com `%%bend nome.bend`. Essa linha salva o conteúdo da célula nesse arquivo e roda o Bend sobre ele. As outras células podem importar o arquivo salvo, por exemplo com `import ./reserva.bend as R`.

Ao rodar um arquivo, o Bend imprime uma destas saídas:

- `ALL PROOFS CHECK`: todas as leis do arquivo foram provadas (ou o arquivo não tem leis e compilou sem erro);
- `SOME PROOFS FAIL`: alguma lei é falsa ou alguma prova está errada. O Bend mostra onde.

Versão completa, com provas de todas as leis: [`colab.ipynb`](https://colab.research.google.com/github/Mateus-Gabardo/bend2-verification-impl/blob/main/colab.ipynb) · [repositório](https://github.com/Mateus-Gabardo/bend2-verification-impl)

## 1. Preparar o ambiente

Esta célula faz três coisas:

1. instala o Bend 2 com o script oficial;
2. cria a pasta de trabalho `/content/bend`;
3. cria o comando `%%bend`, que salva a célula num arquivo e o executa.

Ela só precisa ser executada uma vez por sessão. O estudo usou o Bend **2.0.40**.

In [ ]:
!curl -fsSL https://bend-lang.com/install.sh | sh

import os, subprocess
from IPython.core.magic import register_cell_magic

BEND = os.path.expanduser("~/.bend/bin/bend")
os.makedirs("/content/bend", exist_ok=True)
os.chdir("/content/bend")

@register_cell_magic
def bend(line, cell):
    """%%bend arquivo.bend -> salva a célula nesse arquivo e roda o Bend sobre ele."""
    name = line.strip() or "celula.bend"
    with open(name, "w") as f:
        f.write(cell)
    r = subprocess.run([BEND, name], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())

print(subprocess.run([BEND, "version"], capture_output=True, text=True).stdout)

## 2. O programa (versão correta, v0)

Esta célula tem o programa inteiro. No repositório ele está dividido em quatro módulos, em `src/`; aqui está tudo junto.

- **Tipos:** `Product{id, disponível, reservado}` e o resultado, `Accepted` ou `Refused`.
- **Validação:** `take` retira a quantidade do disponível, ou devolve `None` se faltar estoque.
- **Busca:** `update_where` percorre a lista por recursão e usa a função recebida em `matches` para achar o produto.
- **Reserva:** `reserve` recusa pedido zero e, nos outros casos, chama a busca passando uma função anônima, `p => i => same_id(p, i)`.

O Bend não tem `if`. Toda decisão é feita com `match` (casamento de padrões).

Como o arquivo não tem leis, `ALL PROOFS CHECK` significa apenas que ele compilou sem erro.

In [ ]:
%%bend reserva.bend
# O programa inteiro: tipos, validação, busca e reserva.

import Base

# ---------- tipos ----------

# produto: identificador, disponível e reservado
type Product is Data:
  Product{id: Nat, available: Nat, reserved: Nat}

# resultado: aceita (estoque novo) ou recusada (estoque original)
type Outcome is Data:
  Accepted{stock: List<&2, Product>}
  Refused{stock: List<&2, Product>}

# ---------- validação ----------

# retira n de a: Some{a - n}, ou None{} se faltar estoque
def take(n: Nat, a: Nat) -> Maybe<&2, Nat>:
  match n a:
    case 0n _:
      Some{a}
    case 1n+np 0n:
      None{}
    case 1n+np 1n+ap:
      take(np, ap)

# cria um produto novo com as quantidades atualizadas
def apply_item(p: Product, new_available: Nat, qty: Nat) -> Product:
  match p:
    case Product{id, available, reserved}:
      Product{id, new_available, (reserved + qty : Nat)}

def reserve_item.fin(p: Product, qty: Nat, r: Maybe<&2, Nat>) -> Maybe<&2, Product>:
  match r:
    case Some{left}:
      Some{apply_item(p, left, qty)}
    case None{}:
      None{}

# tenta reservar qty unidades do produto p
def reserve_item(+p: Product, +qty: Nat) -> Maybe<&2, Product>:
  match p:
    case Product{id, available, reserved}:
      reserve_item.fin(p, qty, take(qty, available))

# ---------- busca ----------

def same_id(p: Product, i: Nat) -> Bool:
  match p:
    case Product{id, available, reserved}:
      Nat.is_eq(id, i)

def pick(found: Maybe<&2, List<&2, Product>>, skipped: Maybe<&2, List<&2, Product>>, hit: Bool) -> Maybe<&2, List<&2, Product>>:
  match hit:
    case True{}:
      found
    case False{}:
      skipped

# coloca o produto atualizado na frente do restante da lista
def put_head(r: Maybe<&2, Product>, rest: List<&2, Product>) -> Maybe<&2, List<&2, Product>>:
  match r:
    case Some{p2}:
      Some{p2 <> rest}
    case None{}:
      None{}

def keep_head(p: Product, r: Maybe<&2, List<&2, Product>>) -> Maybe<&2, List<&2, Product>>:
  match r:
    case Some{rest2}:
      Some{p <> rest2}
    case None{}:
      None{}

def update_where(~matches: Product -> Nat -> Bool, stock: List<&2, Product>, +id: Nat, +qty: Nat) -> Maybe<&2, List<&2, Product>>:
  match stock:
    case Nil{}:
      None{}
    case +p <> +rest:
      pick(
        put_head(reserve_item(p, qty), rest),
        keep_head(p, update_where(~matches, rest, id, qty)),
        matches(p, id))

# ---------- reserva ----------

def reserve.fin(stock: List<&2, Product>, r: Maybe<&2, List<&2, Product>>) -> Outcome:
  match r:
    case Some{s2}:
      Accepted{s2}
    case None{}:
      Refused{stock}

def reserve(+stock: List<&2, Product>, +id: Nat, qty: Nat) -> Outcome:
  match qty:
    case 0n:
      Refused{stock}
    case 1n+q:
      reserve.fin(stock, update_where(~(p => i => same_id(p, i)), stock, id, 1n+q))

# ---------- auxiliares para as leis ----------

# soma de disponível + reservado de todos os produtos
def total(xs: List<&2, Product>) -> Nat:
  match xs:
    case Nil{}:
      0n
    case Product{j, a, h} <> t:
      Nat.add(Nat.add(a, h), total(t))

def stock_of(r: Outcome) -> List<&2, Product>:
  match r:
    case Accepted{s}:
      s
    case Refused{s}:
      s

## 3. Executar uma reserva

`main` é a função que o Bend executa. Aqui ela reserva 3 unidades do produto 7, que tem 10 disponíveis e 2 reservadas.

**Esperado:** o produto 7 fica com `7n` disponíveis e `5n` reservadas. O sufixo `n` indica um número natural.

**Experimente:** troque a quantidade (`3n`) ou o produto (`7n`) e rode de novo. O que acontece com `0n`? Com `11n`? Com o produto `9n`?

In [ ]:
%%bend exemplo.bend
import Base
import ./reserva.bend as R

def main() -> R.Outcome:
  R.reserve([R.Product{7n, 10n, 2n}, R.Product{8n, 5n, 0n}], 7n, 3n)

## 4. Testes unitários como leis (T1–T5)

Cada `law` declara uma igualdade `{lado esquerdo == lado direito}`.

- A prova `{==}` pede ao Bend que calcule os dois lados e confira se dão o mesmo valor.
- Num teste unitário, a entrada é fixa (o estoque `sample`). Por isso, a prova é só fazer a conta.

**Esperado:** `ALL PROOFS CHECK`.

**Experimente:** troque o `7n` do resultado de T1 por `6n` e rode. O Bend mostra os dois valores:

- `expected` é o lado esquerdo, isto é, o que o programa calculou;
- `observed` é o lado direito, isto é, o que você escreveu.

In [ ]:
%%bend testes.bend
import Base
import ./reserva.bend as R

def sample() -> List<&2, R.Product>:
  [R.Product{7n, 10n, 2n}, R.Product{8n, 5n, 0n}]

# T1: reserva 3 do produto 7
law t1:
  {R.reserve(sample(), 7n, 3n)
    == R.Accepted{[R.Product{7n, 7n, 5n}, R.Product{8n, 5n, 0n}]} : R.Outcome}
def t1():
  {==}

# T2: reserva 2 do produto 8
law t2:
  {R.reserve(sample(), 8n, 2n)
    == R.Accepted{[R.Product{7n, 10n, 2n}, R.Product{8n, 3n, 2n}]} : R.Outcome}
def t2():
  {==}

# T3: pede 11 do produto 7 (só há 10)
law t3:
  {R.reserve(sample(), 7n, 11n) == R.Refused{sample()} : R.Outcome}
def t3():
  {==}

# T4: pede zero
law t4:
  {R.reserve(sample(), 7n, 0n) == R.Refused{sample()} : R.Outcome}
def t4():
  {==}

# T5: pede o produto 9, que não existe
law t5:
  {R.reserve(sample(), 9n, 1n) == R.Refused{sample()} : R.Outcome}
def t5():
  {==}

## 5. Uma lei geral (P4): vale para qualquer entrada

Aqui `for` significa *para todo*. A lei afirma que, para **qualquer** estoque `s` e **qualquer** produto `i`, pedir zero unidades é recusado e devolve o estoque igual.

Um teste checa um caso; esta lei vale para infinitos casos. Mesmo assim, a prova é simples: `reserve` começa com `match qty: case 0n: Refused{stock}`, então o Bend consegue calcular o resultado sem saber quem são `s` e `i`.

As leis P1 e P2 também são gerais, mas suas provas são longas e exigem indução sobre a lista. Elas estão no `colab.ipynb` e no repositório.

In [ ]:
%%bend lei_geral.bend
import Base
import ./reserva.bend as R

# P4: pedido zero é recusado, para qualquer estoque s e produto i
law p4_refuse_zero:
  for +s : List<&2, R.Product>
  for +i : Nat
  {R.reserve(s, i, 0n) == R.Refused{s} : R.Outcome}

def p4_refuse_zero(s, i):
  {==}

## 6. O caso esgotado

Nenhum dos testes T1–T5 pede **todo** o disponível de um produto. Esta célula cobre esse caso: o produto 7 tem 3 disponíveis e são pedidas as 3.

- **Esgotado, P1 (atualização):** o produto 7 deve continuar na lista, com 0 disponível e 5 reservadas.
- **Esgotado, P2 (conservação):** a soma de disponível + reservado de todo o estoque não muda. Ela é 3+2+5+0 = 10 antes e depois.

**Esperado com a v0:** `ALL PROOFS CHECK`.

In [ ]:
%%bend esgotado.bend
import Base
import ./reserva.bend as R

law esgotado_p1:
  {R.reserve([R.Product{7n, 3n, 2n}, R.Product{8n, 5n, 0n}], 7n, 3n)
    == R.Accepted{[R.Product{7n, 0n, 5n}, R.Product{8n, 5n, 0n}]} : R.Outcome}
def esgotado_p1():
  {==}

law esgotado_p2:
  {R.total(R.stock_of(R.reserve([R.Product{7n, 3n, 2n}, R.Product{8n, 5n, 0n}], 7n, 3n)))
    == R.total([R.Product{7n, 3n, 2n}, R.Product{8n, 5n, 0n}]) : Nat}
def esgotado_p2():
  {==}

## 7. Introduzir o erro da IA (v1)

Na v1, a IA trocou, dentro de `put_head`, a linha

```
Some{p2 <> rest}
```

por

```
Some{keep_if_stocked(p2, rest)}
```

A função `keep_if_stocked` **remove o produto da lista quando o disponível chega a zero**. Junto com ele somem as unidades que acabaram de ser reservadas.

**Para fazer à mão:** na célula 2, troque a linha acima e cole a função abaixo antes de `put_head`. Depois rode a célula 2 de novo.

```
def keep_if_stocked(p: Product, rest: List<&2, Product>) -> List<&2, Product>:
  match p:
    case Product{id, available, reserved}:
      match available:
        case 0n:
          rest
        case 1n+ap:
          Product{id, 1n+ap, reserved} <> rest
```

**Atalho:** a célula abaixo faz essa troca automaticamente no arquivo `reserva.bend`, sem mexer no texto da célula 2. Para voltar à v0, basta rodar a célula 2 de novo.

In [ ]:
src = open("reserva.bend").read()
bug = """# ERRO (versão da IA): remove o produto quando o disponível chega a zero
def keep_if_stocked(p: Product, rest: List<&2, Product>) -> List<&2, Product>:
  match p:
    case Product{id, available, reserved}:
      match available:
        case 0n:
          rest
        case 1n+ap:
          Product{id, 1n+ap, reserved} <> rest

"""
if "keep_if_stocked" not in src:
    src = src.replace("# coloca o produto atualizado", bug + "# coloca o produto atualizado")
    src = src.replace("Some{p2 <> rest}", "Some{keep_if_stocked(p2, rest)}")
    open("reserva.bend", "w").write(src)
print("reserva.bend agora é a v1 (com o erro)." if "keep_if_stocked" in src else "falhou")

## 8. Verificar de novo, agora com a v1

As duas células abaixo são as mesmas das seções 4 e 6, executadas de novo sobre a v1.

- **Testes T1–T5:** `ALL PROOFS CHECK`. Os testes **não** detectam o erro.
- **Caso esgotado:** `SOME PROOFS FAIL`. No `expected`, o produto 7 sumiu da lista e a soma caiu de 10 para 5.

Essa é a conclusão do artigo: os testes conferem apenas os casos escolhidos por quem os escreveu, enquanto as leis gerais P1 e P2 cobrem também o produto esgotado.

**Experimente:** escreva na seção 4 um teste T6 que peça as 10 unidades do produto 7. Ele pega o erro? Ele passa na v0?

In [ ]:
!{BEND} testes.bend

In [ ]:
!{BEND} esgotado.bend